# 🚗 JalanCerdas AI — Colab Training Pipeline

YOLOv8n pothole detection. Run all cells top-to-bottom.

**Prereq:** Runtime → Change runtime type → **T4 GPU**

Steps:
1. Install deps
2. Download dataset (Roboflow)
3. Train YOLOv8n
4. Evaluate
5. Export TFLite
6. Download model

## 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics roboflow

import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  NO GPU! Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("No GPU detected")

## 2 — Download Dataset

Uses Roboflow (free account). Get your API key at https://app.roboflow.com/settings/api

Paste your key in the box below when prompted.

In [ ]:
from getpass import getpass
from roboflow import Roboflow
from pathlib import Path
import yaml

# --- Get Roboflow API key ---
RF_KEY = getpass("Paste Roboflow API key: ")

# --- Download pothole dataset ---
# Using Joseph Nelson's Pothole Detection (YOLOv8 format, well-annotated)
rf = Roboflow(api_key=RF_KEY)
project = rf.workspace("joseph-nelson").project("pothole-detection")
version = project.version(3)  # version 3 — YOLOv8 ready
dataset = version.download("yolov8")

DATASET_DIR = Path(dataset.location)
print(f"\nDataset downloaded to: {DATASET_DIR}")

# --- Show dataset stats ---
for split in ["train", "valid", "test"]:
    img_dir = DATASET_DIR / split
    if img_dir.exists():
        n = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png")))
        print(f"  {split}: {n} images")

# --- Read and patch data.yaml ---
# Roboflow uses 'valid' instead of 'val' — fix for ultralytics
yaml_path = DATASET_DIR / "data.yaml"
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)

# Patch: rename valid -> val if needed
if "valid" in data_cfg and "val" not in data_cfg:
    data_cfg["val"] = data_cfg.pop("valid")
    with open(yaml_path, "w") as f:
        yaml.dump(data_cfg, f, default_flow_style=False)
    print("\nPatched data.yaml: valid → val")

print(f"\nConfig:")
print(f"  Classes: {data_cfg.get('nc')} — {data_cfg.get('names')}")
print(f"  Train:   {data_cfg.get('train')}")
print(f"  Val:     {data_cfg.get('val')}")
print(f"  Test:    {data_cfg.get('test', 'N/A')}")
print(f"\nFull path: {yaml_path}")

## 3 — Train YOLOv8n

100 epochs, batch 16, image 640. ~15-25 min on T4.

In [ ]:
from ultralytics import YOLO
import time

# --- Config ---
MODEL = "yolov8n.pt"     # nano — fast, small, good enough for mobile
EPOCHS = 100
BATCH = 16
IMGSZ = 640
DATA_YAML = str(yaml_path)

# --- Load pretrained model ---
model = YOLO(MODEL)

# --- Train ---
start = time.time()
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=0,
    patience=30,
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    project="runs",
    name="train_colab",
    exist_ok=True,
    verbose=True,
)

elapsed = time.time() - start
h, rem = divmod(elapsed, 3600)
m, s = divmod(rem, 60)

SAVE_DIR = Path(results.save_dir)
BEST_PT = SAVE_DIR / "weights" / "best.pt"

print(f"\n{'='*50}")
print(f"  Training done in {int(h)}h {int(m)}m {int(s)}s")
print(f"  Save dir:  {SAVE_DIR}")
print(f"  Best PT:   {BEST_PT}")
print(f"  Size:      {BEST_PT.stat().st_size / (1024*1024):.2f} MB")
print(f"{'='*50}")

## 4 — Evaluate Model

In [ ]:
val_model = YOLO(str(BEST_PT))
val_results = val_model.val(data=DATA_YAML, device=0)

print(f"\n📊 Validation Results:")
print(f"  mAP@50:    {val_results.box.map50:.4f}")
print(f"  mAP@50-95: {val_results.box.map:.4f}")
print(f"  Precision: {val_results.box.mp:.4f}")
print(f"  Recall:    {val_results.box.mr:.4f}")

f1 = 2 * (val_results.box.mp * val_results.box.mr) / (val_results.box.mp + val_results.box.mr + 1e-8)
print(f"  F1 Score:  {f1:.4f}")

## 5 — Export to TFLite

FP16 quantization. Target: <10 MB for mobile.

In [ ]:
export_model = YOLO(str(BEST_PT))

# FP16 export
print("📦 Exporting to TFLite (FP16)...")
export_path = export_model.export(
    format="tflite",
    imgsz=IMGSZ,
    half=True,
    simplify=True,
)

tflite_file = Path(export_path)
size_mb = tflite_file.stat().st_size / (1024 * 1024)

print(f"\n✅ TFLite exported!")
print(f"  Path: {tflite_file}")
print(f"  Size: {size_mb:.2f} MB")

if size_mb > 10:
    print(f"  ⚠️  Over 10MB — consider INT8 quant for smaller size")
    print(f"  Re-export: export_model.export(format='tflite', int8=True, data=DATA_YAML)")

## 6 — Test Inference

In [ ]:
from IPython.display import display, Image as IPImage
import random

# Find test/val images
test_dir = DATASET_DIR / "test"
val_dir = DATASET_DIR / "valid"
img_dir = test_dir if test_dir.exists() and any(test_dir.glob("*.jpg")) else val_dir

all_imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
samples = random.sample(all_imgs, min(6, len(all_imgs)))

infer = YOLO(str(BEST_PT))
pred_results = infer.predict(
    source=[str(s) for s in samples],
    conf=0.25,
    device=0,
    save=True,
    project="runs",
    name="predict_colab",
    exist_ok=True,
)

print("\n🔍 Inference results:")
for r, s in zip(pred_results, samples):
    boxes = r.boxes
    n_det = len(boxes)
    confs = [f"{float(c):.0%}" for c in boxes.conf] if n_det else []
    print(f"  {s.name}: {n_det} detection(s) {confs}")

print(f"\nResults saved to: runs/predict_colab/")

## 7 — Download Model

Download `best.pt` and `.tflite` to your local machine.

In [ ]:
from google.colab import files

# Download best.pt
print(f"⬇️  Downloading best.pt ({BEST_PT.stat().st_size / (1024*1024):.2f} MB)...")
files.download(str(BEST_PT))

# Download tflite
if tflite_file.exists():
    print(f"⬇️  Downloading {tflite_file.name} ({size_mb:.2f} MB)...")
    files.download(str(tflite_file))

print("\n✅ Done!")
print("\nNext steps:")
print("  1. Copy .tflite to mobile/assets/models/")
print("  2. Build Flutter APK on local machine")
print("  3. Test camera + GPS + detection")

## Summary

| Item | Detail |
|------|--------|
| Model | YOLOv8n (nano) |
| Dataset | Roboflow Pothole Detection v3 |
| Epochs | 100 |
| GPU | T4 (Colab free tier) |
| Export | TFLite FP16 |
| Target | Mobile (Flutter) |

**Artifacts to keep:**
- `best.pt` — full model (for reference / retraining)
- `*.tflite` — mobile model (deploy to Flutter)

**Copy `.tflite` → `mobile/assets/models/best.tflite`**